# Data Handling

Second notebook in the PySpark learning series — continues from 01_read_write_files_operation, using the same UC-volume dataset setup.

   
# Setup — Paths & Imports

Shared constants used across this notebook:

* `setup_data_dirs(dir_name)` — reusable helper: copies `data/{dir_name}` from the repo into a UC volume and creates its matching output volume (`{dir_name}` → `/Volumes/workspace/default/{dir_name}/`, outputs → `/Volumes/workspace/default/{dir_name}-output/`)
* `FLIGHT_DATA_DIR` / `OUTPUT_DIR` — dataset and write-output paths, derived from the helper
* `pprint` — pretty-printer for readable `dtypes` and schema output

In [0]:
import shutil
from pathlib import Path

from pprint import pprint

# Reusable setup — Unit Catalog (UC) volumes are the only workspace locations Spark can both
# READ from and WRITE to. This helper copies a dataset directory from the git
# repo into a UC volume and prepares its matching output directory.
REPO_DATA_ROOT = Path("/Workspace/Repos/mayurkedari16@gmail.com/pyspark-databricks/data")
VOLUME_ROOT = Path("/Volumes/workspace/default")


def setup_data_dirs(
    dir_name: str,
    data_volume: str | None = None,
    output_volume: str | None = None,
) -> tuple[str, str]:
    """
    Copy <repo>/data/{dir_name} into a UC volume and create its output directory.

    * data:   {REPO_DATA_ROOT}/{dir_name}  ->  {VOLUME_ROOT}/{data_volume or dir_name}
    * output:                                 {VOLUME_ROOT}/{output_volume or {dir_name}-output}

    Creates the volumes if they don't exist (CREATE VOLUME IF NOT EXISTS) and is
    idempotent — re-running safely overwrites the staged copies.

    Returns (data_dir, output_dir) as strings; data_dir ends in "/"
    for path concatenation (FLIGHT_DATA_DIR + "csv/*.csv").
    """
    src = Path(REPO_DATA_ROOT) / dir_name
    if not src.is_dir():
        raise FileNotFoundError(f"No dataset directory at {src}")

    data_volume = data_volume or dir_name
    output_volume = output_volume or (dir_name if dir_name.endswith("-output") else f"{dir_name}-output")

    # A volume must exist in Unity Catalog before its /Volumes path is usable
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{data_volume}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{output_volume}`")

    data_dir = Path(VOLUME_ROOT) / data_volume
    output_dir = Path(VOLUME_ROOT) / output_volume

    shutil.copytree(src, data_dir, dirs_exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    return data_dir.as_posix() + "/", output_dir.as_posix() + "/"

In [0]:
import tempfile

csv_data = """id,name,age,salary,address,nominee
1,Manish,26,75000,Bihar,Suresh Kumar
2,Nikita,23,100000,Uttar Pradesh,Ramesh Singh
3,Pritam,22,150000,Bangalore,India,Anjali Sharma
4,Prantosh,17,200000,Kolkata,India,Rahul Das
5,Vikash,31,300000,,Pooja Gupta"""


In [0]:
# %fs mkdirs "/Volumes/workspace/default/emp-data-output"

# %sql "CREATE VOLUME IF NOT EXISTS workspace.default.`emp-data-output`"

In [0]:
# Wire up this notebook's dataset using the reusable helper — copies
# data/flight-data from the repo into the flight_data volume (idempotent)
# and points OUTPUT_DIR at the flight_output volume.
EMP_DATA_DIR = f"{VOLUME_ROOT}/emp-data-output/"
OUTPUT_DIR = EMP_DATA_DIR
with open(EMP_DATA_DIR + "emp-data.csv", mode='w') as f:
    f.write(csv_data)
# EMP_DATA_DIR, OUTPUT_DIR = setup_data_dirs(
#     "emp-data",
#     data_volume="emp-data",
#     output_volume="emp-data",
# )

print("Dataset Dir:", EMP_DATA_DIR)
print("Outputs Dir:", OUTPUT_DIR)

# CORRUPTED / MALFORMED DATA HANDLING

In [0]:

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark Data Handling").getOrCreate()


In [0]:
# Fetches the globally configured column name for corrupt record tracking
try:
    print(spark.conf.get("spark.sql.columnNameOfCorruptRecord"))
except Exception:
    print("_corrupt_record")


In [0]:
from pyspark.sql.types import (
    StructField, 
    StructType, 
    StringType, 
    IntegerType,
    DoubleType
)

employee_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("address", StringType(), True),
    StructField("nominee", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

employees_permissive_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE") # all 5 rows are valid
    .schema(employee_schema)
    .load(EMP_DATA_DIR)
)

employees_permissive_df.show()


--------------------------------------------------
--------------------------------------------------

In [0]:
employees_dropmalformed_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "DROPMALFORMED") # 2 malformed rows are dropped, 3 valid rows remain
    .schema(employee_schema)
    .load(EMP_DATA_DIR)
)

employees_dropmalformed_df.show()

In [0]:
# This code fails due to having corrupted/malformed records

employees_failfast_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST") # Execution fails dur to 2 malformed rows
    .schema(employee_schema)
    .load(EMP_DATA_DIR)
)

employees_failfast_df.show()

## Print Bad/Corrupt/Malform Records

Add an additional `StringType` column to the schema — Spark populates it with the raw malformed row, or `NULL` if the row is valid. The column name can be anything, but the standard convention is `_corrupt_record`.

**Note:** If the column name is anything other than `_corrupt_record`, Spark drops non-malformed records and keeps only the malformed ones. This is controlled by the `spark.sql.columnNameOfCorruptRecord` property, which defaults to `_corrupt_record` out-of-the-box in Apache Spark.

**Parse modes:**

| Mode | Behavior |
| --- | --- |
| `PERMISSIVE` | Keeps bad rows; sets malformed fields to `NULL`, raw row goes to `_corrupt_record` |
| `DROPMALFORMED` | Silently drops bad rows |
| `FAILFAST` | Throws an exception on the first bad row |

**Example:**
```python

employee_schema_display_corrupt_records = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("address", StringType(), True),
    StructField("nominee", StringType(), True),
    StructField("_corrupt_record", StringType(), True), # captures malformed rows
])
```

In [0]:
employee_schema_display_corrupt_records = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("address", StringType(), True),
    StructField("nominee", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

employee_schema_display_corrupt_records_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .schema(employee_schema_display_corrupt_records)
    .load(EMP_DATA_DIR)
)

employee_schema_display_corrupt_records_df.show()

## Saving Bad Records in a File

Set `badRecordsPath` to a directory path — Spark writes corrupt rows there instead of failing.

**Notes:**
* When `badRecordsPath` is set, `mode` must **not** be specified — Spark throws an error if both are present.
* Bad records are always stored as a **JSON file** inside the `badRecordsPath` directory.

**Example:**
```python
.option("badRecordsPath", f"{OUTPUT_DIR}/bad_records/")
```

In [0]:
# This code fails due to having corrupted/malformed records

employees_failfast_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("badRecordsPath", EMP_DATA_DIR + "bad_records/")
    .schema(employee_schema)
    .load(EMP_DATA_DIR)
)

employees_failfast_df.show()

In [0]:
print(EMP_DATA_DIR + "bad_records/")

In [0]:
%fs
ls /Volumes/workspace/default/emp-data-output/

# Print the contents of the badRecordsPath directory

In [0]:
%sh

cat /Volumes/workspace/default/emp-data-output/bad_records/20261004T113625/bad_records/part-00000-e39f3551-1825-41f7-9168-3de1f7b80035

In [0]:
bad_records_df = (
    spark.read.format("json")
    .option("inferSchema", True)
    .load(EMP_DATA_DIR + "bad_records/20261004T113625/bad_records/*")
)

bad_records_df.show()